# `_template` — the certification notebook skeleton

**This notebook is not a certification.** It is the nine-section template every scenario
notebook `C<nn>_<slug>.ipynb` follows, and the harness exercise the merge gate runs twice on every
gate (the two cycle dry runs of `tests/omiecid/certification/test_cert_dry_runs.py`). Copy it, keep
the section structure and the cell tags (`section-<k>`, `parameters`, `section-9-render`), and
replace the contents of each section.

## 1. Purpose

*(In a real scenario: the purpose paragraph of the scenario, verbatim from
[`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md): what it pins, why it matters, and
what a failure would implicate.)*

The template's own purpose is narrow: draw the single crossing pair (`C02.crossing_pair`) and
check that the one trade prints at **the resting order's limit price**, for
$\min(q_{rest}, q_{agg})$. That is C02's claim in miniature, small enough to read and large
enough to exercise the whole harness: the quote checker, the generator, the record builder
(eager and jitted), the invariant battery, the results JSON and section 9.

## 2. The rule, quoted

The governing passage, quoted **verbatim** and checked against its source document at run time
(`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged
**DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends
`NOT CERTIFIED`.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Maker pricing: the trade prints at the resting order's limit, never the aggressor's.",
    ),
)
CITATIONS = ()

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it
(`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next
cell. `INJECT_FAILURE = True` moves the **expectation** by one price tick (the engine is
untouched): it is how the deliberately failed dry run is produced.

In [ ]:
CERT_ID = "C99"          # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 3                # a real scenario states its own draw count
SEED0 = 0                # first seed
INJECT_FAILURE = False   # True: expected trade price = resting price + 1 tick (dry-run failure)

In [ ]:
import pathlib

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig

CONSTRUCTION = "C02.crossing_pair"
CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "_template.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}, {DRAWS} draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts)")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW")
print(f"capacity envelope: {ENVELOPE}")
print(f"INJECT_FAILURE = {INJECT_FAILURE}")

## 4. Dataset

Build the tapes, then **look at them**: readable units (EUR/MWh, MW) beside the raw integer
encoding, and the book shape of the first draw.

In [ ]:
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

DRAWN = {s: synth.CONSTRUCTIONS[CONSTRUCTION](s) for s in range(SEED0, SEED0 + DRAWS)}
first = DRAWN[SEED0]

rows = ["| oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | ts |",
        "|---|---|---|---|---|---|---|---|"]
for r in first.tape[first.tape[:, synth.COL_OID] >= 1]:
    rows.append(
        f"| {int(r[synth.COL_OID])} | {'BUY' if int(r[synth.COL_SIDE]) == 0 else 'SELL'} | "
        f"{decode_price(int(r[synth.COL_PRICE]), CONFIG):.2f} | {decode_qty(int(r[synth.COL_QTY]), CONFIG):.1f} | "
        f"{int(r[synth.COL_PRICE])} | {int(r[synth.COL_QTY])} | {int(r[synth.COL_AGENT])} | "
        f"{int(r[synth.COL_TS])} |")
display(Markdown(f"**Seed {SEED0}** (digest `{first.digest()[:16]}…`)\n\n" + "\n".join(rows)))
for s, d in DRAWN.items():
    p = d.params
    print(f"seed {s}: rest {p['p_rest_eur']:.2f} EUR/MWh x {p['q_rest_mw']:.1f} MW (int {p['p_rest_int']} / "
          f"{p['q_rest_int']}), aggressor {p['p_agg_eur']:.2f} x {p['q_agg_mw']:.1f} (gap {p['gap_ticks']} ticks)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.6))
for r in first.tape[first.tape[:, synth.COL_OID] >= 1]:
    buy = int(r[synth.COL_SIDE]) == 0
    ax.barh(decode_price(int(r[synth.COL_PRICE]), CONFIG), decode_qty(int(r[synth.COL_QTY]), CONFIG) * (1 if buy else -1),
            height=0.6, color="tab:green" if buy else "tab:red", label="BUY" if buy else "SELL")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("quantity (MW): buy to the right, sell to the left")
ax.set_ylabel("limit price (EUR/MWh)")
ax.set_title(f"{CONSTRUCTION}, seed {SEED0}: the order book shape")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 5. Known result

For a resting order at limit price $p_{rest}$ and quantity $q_{rest}$, and an aggressor priced
through it for $q_{agg}$, exactly one trade occurs, of quantity
$q = \min(q_{rest},\, q_{agg})$, **at** $p_{rest}$ (maker pricing, §6.2 item 3).

This is a *parametric golden*: the dataset is random and the answer is still known, because it is
a formula over the draw's own parameters (`draw.params`: `p_rest_int`, `q_rest_int`, `q_agg_int`),
in integer ticks. With `INJECT_FAILURE = True` the expected price is $p_{rest} + 1$ tick.

In [ ]:
def expectation(draw):
    # The oracle, from the draw's parameters (never from the generator's own expectation).
    p = draw.params
    price = int(p["p_rest_int"]) + (1 if INJECT_FAILURE else 0)
    return {"n_trades": 1, "trade_price_int": price,
            "trade_qty_int": min(int(p["q_rest_int"]), int(p["q_agg_int"]))}


for s, d in DRAWN.items():
    e = expectation(d)
    print(f"seed {s}: 1 trade at price_int {e['trade_price_int']} "
          f"({decode_price(e['trade_price_int'], CONFIG):.2f} EUR/MWh) for qty_int {e['trade_qty_int']} "
          f"({decode_qty(e['trade_qty_int'], CONFIG):.1f} MW)")

## 6. Run

Every draw goes through the record builder twice, **eager** (`jit=False`, under
`jax.disable_jit()`) and **jitted** (`jit=True`), and the two records must agree on every integer
field (`compare_records`). Jitted is the mode production uses.

In [ ]:
from tests.omiecid.reference.omie_invariants import compare_records, record_from_draw


def run_draw(seed):
    draw = DRAWN.get(seed) or synth.CONSTRUCTIONS[CONSTRUCTION](seed)
    eager = record_from_draw(draw, jit=False)
    jitted = record_from_draw(draw, jit=True)
    return draw, eager, jitted, compare_records(eager, jitted)


RUNS = {s: run_draw(s) for s in range(SEED0, SEED0 + DRAWS)}
for s, (_, eager, jitted, diff) in RUNS.items():
    print(f"seed {s}: {len(jitted.fills)} fill(s); eager == jitted: {diff == ()}"
          + (f" (differs on {', '.join(diff)})" if diff else ""))

## 7. Certification

The oracle comparison in integer ticks, then the invariant battery (RB, I1–I9) as a pass/fail
table, the capacity peaks and the discard rate. A draw the battery marks I8 `invalid` (outside the
capacity envelope) is discarded and replaced by the next seed. No check stops the notebook: every
assertion is evaluated and recorded.

In [ ]:
from tests.omiecid.reference.omie_invariants import run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Template: the single crossing pair",
    known_result="one trade; p_trade = p_rest; q_trade = min(q_rest, q_agg)",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=(CONSTRUCTION,),
    requested_draws=DRAWS, runtime_forecast_s=30.0, started_at=T0,
    downstream="every scenario that relies on maker pricing (C02, C05, C06, C08-C10) and invariant I5",
)

valid, seed = 0, SEED0
while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
    draw, eager, jitted, diff = RUNS.get(seed) or run_draw(seed)
    report = run_battery(jitted, expects_trade=True)
    if not cert.add_battery(report, seed=seed):
        seed += 1
        continue
    valid += 1
    e = expectation(draw)
    fills = jitted.fills
    cert.record_case(seed, draw.tape, draw.params)
    cert.check_equal(f"seed {seed}: number of trades", len(fills), e["n_trades"], seed=seed, unit="count",
                     rule="§6.2 item 3")
    cert.check_equal(f"seed {seed}: trade price (maker)", fills[0].price_int if fills else None,
                     e["trade_price_int"], seed=seed, unit="price_int", rule="§6.2 item 3")
    cert.check_equal(f"seed {seed}: trade quantity", sum(f.qty_int for f in fills), e["trade_qty_int"],
                     seed=seed, unit="qty_int", rule="§6.2 item 3")
    cert.set_eager_jit(seed, diff == (), detail=", ".join(diff))
    cert.record_capacity({**jitted.observed_capacity, "hops": None})   # hops: unobserved by the engine
    seed += 1

display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))